In [3]:
import pandas as pd

In [14]:
bus_data = pd.read_csv("../../data/A0008D - v_q_bus_stop (full).csv")
bus_location_data = pd.read_csv("../../data/correct_bus_location.csv")
bus_additional_data = pd.read_csv("../../data/A0008D - v_q_vls_marker (full).csv")

In [15]:
bus_consolidated = bus_data.merge(
    bus_location_data,
    how='outer',
    left_on='BUS_STOP_CD',
    right_on='BusStopCode'
)

bus_needed = bus_consolidated[[
    "BUS_STOP_CD",
    "BUS_STOP_NAM",
    "MRK_ID_NUM",
    "BusStopCode",
    "Description",
    "Latitude",
    "Longitude"
]].copy()

bus_needed = bus_needed.rename(columns={
    "BUS_STOP_NAM": "STATION_NAME",
    "Latitude": "LATITUDE",
    "Longitude": "LONGITUDE"
})

bus_needed = bus_needed.rename(columns={
    "BUS_STOP_NAM": "STATION_NAME",
    "Description": "LTA_DESCRIPTION",
    "Latitude": "LATITUDE",
    "Longitude": "LONGITUDE"
})
bus_needed["FINAL_STOP_CODE"] = bus_needed["BUS_STOP_CD"].fillna(bus_needed["BusStopCode"])
bus_needed["STATION_NAME"] = bus_needed["STATION_NAME"].fillna(bus_needed["LTA_DESCRIPTION"])
bus_needed['Travel_Type'] = 'BUS'

bus_needed_final = bus_needed[[
    "FINAL_STOP_CODE",
    "STATION_NAME",
    "MRK_ID_NUM",
    "LATITUDE",
    "LONGITUDE",
    "Travel_Type"
]].copy()

bus_additional_data["LATITUDE"] = bus_additional_data["LATITUDE_VAL"] / 3600000
bus_additional_data["LONGITUDE"] = bus_additional_data["LONGITUDE_VAL"] / 3600000

bus_additional_data_clean = bus_additional_data[[
    "MRK_ID_NUM",
    "LATITUDE",
    "LONGITUDE"
]].copy()

bus_needed_final = bus_needed_final.merge(
    bus_additional_data_clean,
    on="MRK_ID_NUM",
    how="left",
    suffixes=("", "_additional")
)


bus_needed_final["LATITUDE"] = bus_needed_final["LATITUDE"].fillna(
    bus_needed_final["LATITUDE_additional"]
)

bus_needed_final["LONGITUDE"] = bus_needed_final["LONGITUDE"].fillna(
    bus_needed_final["LONGITUDE_additional"]
)

bus_needed_final = bus_needed_final.drop(columns=[
    "LATITUDE_additional",
    "LONGITUDE_additional"
])

print("Remaining missing coords:",
      bus_needed_final["LATITUDE"].isna().sum())

Remaining missing coords: 0


In [16]:
print(bus_consolidated.columns)

Index(['BUS_STOP_CD', 'BUS_STOP_NAM', 'MRK_ID_NUM', 'RD_NAM', 'BusStopCode',
       'RoadName', 'Description', 'Latitude', 'Longitude'],
      dtype='str')


In [17]:
train_data = pd.read_csv('../../data/mrt_station_coordinates.csv')
train_needed = train_data[['NODE_NAME', 'NODE_MRK_ID', 'LATITUDE', 'LONGITUDE']]
train_needed = train_needed.rename(columns={
    'NODE_NAME': 'STATION_NAME',
    'NODE_MRK_ID': 'MRK_ID_NUM'
})
train_needed['Travel_Type'] = 'TRAIN'
train_needed.head(5)

,STATION_NAME,MRK_ID_NUM,LATITUDE,LONGITUDE,Travel_Type
0,Yio Chu Kang,1,1.381756,103.844947,TRAIN
1,Ang Mo Kio,2,1.369933,103.849558,TRAIN
2,Bishan NSEW,3,1.350839,103.848144,TRAIN
3,Braddell,4,1.340469,103.846799,TRAIN
4,Toa Payoh,5,1.332629,103.847502,TRAIN


In [18]:
bus_needed_final = bus_needed_final[[
    "STATION_NAME",
    "MRK_ID_NUM",
    "LATITUDE",
    "LONGITUDE",
    "Travel_Type"
]]

consolidated_stations = pd.concat(
    [bus_needed_final, train_needed],
    axis=0,
    ignore_index=True
)

consolidated_stations.tail(5)

,STATION_NAME,MRK_ID_NUM,LATITUDE,LONGITUDE,Travel_Type
6271,Marine Parade,427.0,1.302865,103.905508,TRAIN
6272,Marine Terrace,428.0,1.306786,103.915316,TRAIN
6273,Siglap,429.0,1.309877,103.929879,TRAIN
6274,Bayshore,430.0,1.312841,103.941597,TRAIN
6275,Hume,336.0,1.354511,103.769104,TRAIN


In [19]:
df = pd.read_csv('../../data/A0008D - v_pt_ride_all (full).csv')

In [20]:
df['BIZ_DT'] = pd.to_datetime(df['BIZ_DT'])
df['ENTRY_DT'] = pd.to_datetime(df['ENTRY_DT'])
df['EXIT_DT'] = pd.to_datetime(df['EXIT_DT'])
df['ENTRY_TM'] = pd.to_datetime(
    df['ENTRY_DT'].astype(str) + ' ' + df['ENTRY_TM'],
    format='%Y-%m-%d %H:%M:%S.%f'
)
df['EXIT_TM'] = pd.to_datetime(
    df['EXIT_DT'].astype(str) + ' ' + df['EXIT_TM'],
    format='%Y-%m-%d %H:%M:%S.%f'
)

In [21]:
df.dtypes

BIZ_DT                datetime64[us]
BUS_SVC_NUM                  float64
CRD_NUM                        int64
DEST_LOC_ID_NUM                int64
ENTRY_DT              datetime64[us]
ENTRY_TM              datetime64[us]
EXIT_DT               datetime64[us]
EXIT_TM               datetime64[us]
JRNY_ID_NUM                    int64
ORIG_LOC_ID_NUM                int64
PATRON_CATG_ID_NUM             int64
PAY_CD                         int64
RIDE_DISC_AMT                float64
RIDE_DIST_KM_CNT             float64
RIDE_FARE_AMT                float64
RIDE_ID_NUM                    int64
RIDE_MIN_CNT                 float64
RIDE_ST_CD                     int64
SVC_GRADE_ID_NUM               int64
TKT_TYP_CD                     int64
TRNSPT_MODE_CD                 int64
dtype: object

In [22]:
df = df.merge(consolidated_stations, how = 'left', left_on= 'DEST_LOC_ID_NUM',
                right_on= 'MRK_ID_NUM')
df.head(5)

,BIZ_DT,BUS_SVC_NUM,CRD_NUM,DEST_LOC_ID_NUM,ENTRY_DT,ENTRY_TM,EXIT_DT,EXIT_TM,JRNY_ID_NUM,ORIG_LOC_ID_NUM,...,RIDE_MIN_CNT,RIDE_ST_CD,SVC_GRADE_ID_NUM,TKT_TYP_CD,TRNSPT_MODE_CD,STATION_NAME,MRK_ID_NUM,LATITUDE,LONGITUDE,Travel_Type
0,2025-02-22,NaN,9200002624150,402,2025-02-22,2025-02-22 12:37:05,2025-02-22,2025-02-22 12:37:29,110780691633,403,...,0.400,1,2,39,2,Woodlands TEL,402.0,1.436058,103.787939,TRAIN
1,2025-02-22,187.0,240000373709,2753,2025-02-22,2025-02-22 10:05:07,2025-02-22,2025-02-22 10:09:32,110780657323,2767,...,4.417,1,1,41,1,Opp Blk 628,2753.0,1.351765,103.750199,BUS
2,2025-02-22,168.0,9200002624150,4344,2025-02-22,2025-02-22 12:46:47,2025-02-22,2025-02-22 13:15:51,110780691633,5060,...,29.067,1,1,39,1,Bef Seletar Camp G,4344.0,1.402222,103.871388,BUS
3,2025-02-22,NaN,240000373709,17,2025-02-22,2025-02-22 10:40:23,2025-02-22,2025-02-22 10:41:11,110780657323,28,...,0.800,1,1,41,2,Redhill,17.0,1.289635,103.816741,TRAIN
4,2025-02-22,NaN,9190001883020,41,2025-02-22,2025-02-22 10:30:17,2025-02-22,2025-02-22 10:39:53,110780692295,42,...,9.600,1,1,40,2,Tampines NSEW,41.0,1.353302,103.945145,TRAIN


In [24]:
df.rename(columns={
    'STATION_NAME': 'DEST_STATION_NAME',
    'MRK_ID_NUM': 'DEST_MRK_ID_NUM',
    'LATITUDE': 'DEST_LATITUDE',
    'LONGITUDE': 'DEST_LONGITUDE',
    'Travel_Type': 'DEST_Travel_Type'
}, inplace=True)

In [25]:
df = df.merge(consolidated_stations, how = 'left', left_on= 'ORIG_LOC_ID_NUM',
                right_on= 'MRK_ID_NUM')

In [26]:
df.rename(columns={
    'STATION_NAME': 'ORIG_STATION_NAME',
    'MRK_ID_NUM': 'ORIG_MRK_ID_NUM',
    'LATITUDE': 'ORIG_LATITUDE',
    'LONGITUDE': 'ORIG_LONGITUDE',
    'Travel_Type': 'ORIG_Travel_Type'
}, inplace=True)
df.head(5)

,BIZ_DT,BUS_SVC_NUM,CRD_NUM,DEST_LOC_ID_NUM,ENTRY_DT,ENTRY_TM,EXIT_DT,EXIT_TM,JRNY_ID_NUM,ORIG_LOC_ID_NUM,...,DEST_STATION_NAME,DEST_MRK_ID_NUM,DEST_LATITUDE,DEST_LONGITUDE,DEST_Travel_Type,ORIG_STATION_NAME,ORIG_MRK_ID_NUM,ORIG_LATITUDE,ORIG_LONGITUDE,ORIG_Travel_Type
0,2025-02-22,NaN,9200002624150,402,2025-02-22,2025-02-22 12:37:05,2025-02-22,2025-02-22 12:37:29,110780691633,403,...,Woodlands TEL,402.0,1.436058,103.787939,TRAIN,Woodlands South,403.0,1.427396,103.793264,TRAIN
1,2025-02-22,187.0,240000373709,2753,2025-02-22,2025-02-22 10:05:07,2025-02-22,2025-02-22 10:09:32,110780657323,2767,...,Opp Blk 628,2753.0,1.351765,103.750199,BUS,Blk 315,2767.0,1.359929,103.746379,BUS
2,2025-02-22,168.0,9200002624150,4344,2025-02-22,2025-02-22 12:46:47,2025-02-22,2025-02-22 13:15:51,110780691633,5060,...,Bef Seletar Camp G,4344.0,1.402222,103.871388,BUS,Woodlands Int B10,5060.0,1.436946,103.785936,BUS
3,2025-02-22,NaN,240000373709,17,2025-02-22,2025-02-22 10:40:23,2025-02-22,2025-02-22 10:41:11,110780657323,28,...,Redhill,17.0,1.289635,103.816741,TRAIN,Bukit Batok,28.0,1.349033,103.749567,TRAIN
4,2025-02-22,NaN,9190001883020,41,2025-02-22,2025-02-22 10:30:17,2025-02-22,2025-02-22 10:39:53,110780692295,42,...,Tampines NSEW,41.0,1.353302,103.945145,TRAIN,Pasir Ris,42.0,1.373043,103.949285,TRAIN


In [31]:
df = df[df["ENTRY_DT"].dt.date == pd.Timestamp("2025-02-11").date()].copy()
df3 = df.sort_values(["CRD_NUM", "ENTRY_DT", "ENTRY_TM"], ascending= [True, True, True]).reset_index(drop= True)

In [34]:
# Onemap API does not allow dates more than a year
df3["ENTRY_DT"] = pd.Timestamp("10-04-2025")
df3["EXIT_DT"] = pd.Timestamp("10-04-2025")


df3["ENTRY_TM"] = df3["ENTRY_TM"].dt.strftime("%H:%M:%S")
df3["EXIT_TM"] = df3["EXIT_TM"].dt.strftime("%H:%M:%S")
df3["ENTRY_DT"] = df3["ENTRY_DT"].dt.strftime("%m-%d-%Y")
df3["EXIT_DT"] = df3["EXIT_DT"].dt.strftime("%m-%d-%Y")

In [35]:
df3["next_orig_lat"] = df3.groupby("CRD_NUM")["ORIG_LATITUDE"].shift(-1)
df3["next_orig_lon"] = df3.groupby("CRD_NUM")["ORIG_LONGITUDE"].shift(-1)
df3["next_orig_station"] = df3.groupby("CRD_NUM")["ORIG_STATION_NAME"].shift(-1)

In [36]:
walk_cache = pd.read_csv('../../data/walk_distance_cache.csv')

In [37]:
df3["pair_key"] = (
   df3["DEST_LATITUDE"].astype(str) + "_" +
   df3["DEST_LONGITUDE"].astype(str) + "_" +
   df3["next_orig_lat"].astype(str) + "_" +
   df3["next_orig_lon"].astype(str)
)


pairs = df3[
    [
        "pair_key",
        "DEST_STATION_NAME",
        "next_orig_station",
        "DEST_LATITUDE",
        "DEST_LONGITUDE",
        "next_orig_lat",
        "next_orig_lon"
    ]
].dropna(subset=[
    "DEST_LATITUDE",
    "DEST_LONGITUDE",
    "next_orig_lat",
    "next_orig_lon"
]).drop_duplicates("pair_key").copy()

In [38]:
df3 = (
    df3.drop(columns=["pair_key"], errors="ignore")
    .merge(
        walk_cache.drop(columns=["pair_key"], errors="ignore")[[
            "DEST_LATITUDE", "DEST_LONGITUDE", "next_orig_lat", "next_orig_lon", "walk_distance"
        ]].drop_duplicates(
            subset=["DEST_LATITUDE", "DEST_LONGITUDE", "next_orig_lat", "next_orig_lon"]
        ),
        on=["DEST_LATITUDE", "DEST_LONGITUDE", "next_orig_lat", "next_orig_lon"],
        how="left"
    )
)

In [39]:
# restore the actual filtered business date
real_date = pd.Timestamp("2025-02-11")

df3["ENTRY_DT"] = real_date
df3["EXIT_DT"] = real_date

# if ENTRY_TM / EXIT_TM are currently strings like '00:47:26',
# convert them back into full datetimes
df3["ENTRY_TM"] = pd.to_datetime(
    df3["ENTRY_DT"].astype(str) + " " + df3["ENTRY_TM"].astype(str),
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

df3["EXIT_TM"] = pd.to_datetime(
    df3["EXIT_DT"].astype(str) + " " + df3["EXIT_TM"].astype(str),
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

In [44]:
print(df3.columns)

Index(['BIZ_DT', 'BUS_SVC_NUM', 'CRD_NUM', 'DEST_LOC_ID_NUM', 'ENTRY_DT',
       'ENTRY_TM', 'EXIT_DT', 'EXIT_TM', 'JRNY_ID_NUM', 'ORIG_LOC_ID_NUM',
       'PATRON_CATG_ID_NUM', 'PAY_CD', 'RIDE_DISC_AMT', 'RIDE_DIST_KM_CNT',
       'RIDE_FARE_AMT', 'RIDE_ID_NUM', 'RIDE_MIN_CNT', 'RIDE_ST_CD',
       'SVC_GRADE_ID_NUM', 'TKT_TYP_CD', 'TRNSPT_MODE_CD', 'DEST_STATION_NAME',
       'DEST_MRK_ID_NUM', 'DEST_LATITUDE', 'DEST_LONGITUDE',
       'DEST_Travel_Type', 'ORIG_STATION_NAME', 'ORIG_MRK_ID_NUM',
       'ORIG_LATITUDE', 'ORIG_LONGITUDE', 'ORIG_Travel_Type', 'next_orig_lat',
       'next_orig_lon', 'next_orig_station', 'walk_distance'],
      dtype='str')


In [57]:
df3.head(20)

,BIZ_DT,BUS_SVC_NUM,CRD_NUM,DEST_LOC_ID_NUM,ENTRY_DT,ENTRY_TM,EXIT_DT,EXIT_TM,JRNY_ID_NUM,ORIG_LOC_ID_NUM,...,DEST_Travel_Type,ORIG_STATION_NAME,ORIG_MRK_ID_NUM,ORIG_LATITUDE,ORIG_LONGITUDE,ORIG_Travel_Type,next_orig_lat,next_orig_lon,next_orig_station,walk_distance
0,2025-02-11,NaN,100005879220,22,2025-02-11,2025-02-11 11:20:34,2025-02-11,2025-02-11 11:32:25,110710909992,44,...,TRAIN,Admiralty,44.0,1.440589,103.800990,TRAIN,1.429443,103.835005,Yishun,0.0
1,2025-02-11,NaN,100005879220,44,2025-02-11,2025-02-11 22:32:59,2025-02-11,2025-02-11 22:45:03,110710523649,22,...,TRAIN,Yishun,22.0,1.429443,103.835005,TRAIN,NaN,NaN,NaN,NaN
2,2025-02-11,45.0,100006223599,3968,2025-02-11,2025-02-11 06:56:13,2025-02-11,2025-02-11 07:07:33,110710081977,4009,...,BUS,Blk 172,4009.0,1.350814,103.889844,BUS,1.349708,103.873575,Serangoon NEL,195.0
3,2025-02-11,NaN,100006223599,109,2025-02-11,2025-02-11 07:09:17,2025-02-11,2025-02-11 07:18:09,110710081977,106,...,TRAIN,Serangoon NEL,106.0,1.349708,103.873575,TRAIN,1.319336,103.861570,Boon Keng,0.0
4,2025-02-11,NaN,100006223599,106,2025-02-11,2025-02-11 14:51:22,2025-02-11,2025-02-11 15:03:53,110710904744,109,...,TRAIN,Boon Keng,109.0,1.319336,103.861570,TRAIN,1.348979,103.872774,S'Goon Stn Exit B,139.0
5,2025-02-11,45.0,100006223599,4008,2025-02-11,2025-02-11 15:10:02,2025-02-11,2025-02-11 15:21:55,110710226201,3967,...,BUS,S'Goon Stn Exit B,3967.0,1.348979,103.872774,BUS,NaN,NaN,NaN,NaN
6,2025-02-11,382.0,130013244516,5960,2025-02-11,2025-02-11 10:07:29,2025-02-11,2025-02-11 10:11:38,110710535706,5953,...,BUS,Blk 322 CP,5953.0,1.410690,103.897592,BUS,1.408195,103.905669,One Punggol,123.0
7,2025-02-11,382.0,130013244516,5954,2025-02-11,2025-02-11 12:15:49,2025-02-11,2025-02-11 12:21:50,110709913132,5959,...,BUS,One Punggol,5959.0,1.408195,103.905669,BUS,NaN,NaN,NaN,NaN
8,2025-02-11,871.0,130013244638,2782,2025-02-11,2025-02-11 06:40:58,2025-02-11,2025-02-11 06:54:37,110711365561,6011,...,BUS,Tengah CC,6011.0,1.356473,103.734424,BUS,1.358612,103.751791,Bukit Gombak,248.0
9,2025-02-11,NaN,130013244638,45,2025-02-11,2025-02-11 06:58:38,2025-02-11,2025-02-11 07:20:00,110711365561,29,...,TRAIN,Bukit Gombak,29.0,1.358612,103.751791,TRAIN,1.436946,103.785936,Woodlands Int B5,25.0


Getting planning area dataset
- please go to https://www.onemap.gov.sg/apidocs/planningarea/#planningAreaPolygon and get /api/public/popapi/getAllPlanningarea for year 2019 which is the latest available year

In [1]:
#put code to call onemap api here using your own token 
data = response.json()

NameError: name 'response' is not defined

In [50]:
print(type(data))
print(data.keys())

<class 'dict'>
dict_keys(['SearchResults'])


In [51]:
records = data['SearchResults']

In [52]:
areas = pd.DataFrame(records)

In [53]:
import json
areas['geojson_parsed'] = areas['geojson'].apply(json.loads)

In [54]:
from shapely.geometry import shape
import geopandas as gpd

areas['geometry'] = areas['geojson_parsed'].apply(shape)

gdf = gpd.GeoDataFrame(areas, geometry='geometry', crs="EPSG:4326")

In [56]:
gdf = gdf[['pln_area_n', 'geometry']]
gdf = gdf.rename(columns={'pln_area_n': 'planning_area'})

In [64]:
gdf.head(5)

,planning_area,geometry
0,BEDOK,"MULTIPOLYGON (((103.93208 1.30555, 103.93208 1..."
1,BUKIT TIMAH,"MULTIPOLYGON (((103.79766 1.34813, 103.79806 1..."
2,BUKIT BATOK,"MULTIPOLYGON (((103.76408 1.37001, 103.76444 1..."
3,BUKIT MERAH,"MULTIPOLYGON (((103.82361 1.26018, 103.82362 1..."
4,CENTRAL WATER CATCHMENT,"MULTIPOLYGON (((103.80702 1.41126, 103.80754 1..."


In [67]:
first_tap = df3.groupby('CRD_NUM', as_index=False).first()

#first_tap.head(20)


In [66]:
#first_tap.tail(20)

In [62]:
#safety check
duplicates = first_tap[first_tap.duplicated(subset='CRD_NUM', keep=False)]
print(duplicates)

Empty DataFrame
Columns: [CRD_NUM, BIZ_DT, BUS_SVC_NUM, DEST_LOC_ID_NUM, ENTRY_DT, ENTRY_TM, EXIT_DT, EXIT_TM, JRNY_ID_NUM, ORIG_LOC_ID_NUM, PATRON_CATG_ID_NUM, PAY_CD, RIDE_DISC_AMT, RIDE_DIST_KM_CNT, RIDE_FARE_AMT, RIDE_ID_NUM, RIDE_MIN_CNT, RIDE_ST_CD, SVC_GRADE_ID_NUM, TKT_TYP_CD, TRNSPT_MODE_CD, DEST_STATION_NAME, DEST_MRK_ID_NUM, DEST_LATITUDE, DEST_LONGITUDE, DEST_Travel_Type, ORIG_STATION_NAME, ORIG_MRK_ID_NUM, ORIG_LATITUDE, ORIG_LONGITUDE, ORIG_Travel_Type, next_orig_lat, next_orig_lon, next_orig_station, walk_distance]
Index: []

[0 rows x 35 columns]


In [68]:
# Convert the separate lat/lon columns into point geometries
df_points = gpd.GeoDataFrame(
    first_tap, 
    geometry=gpd.points_from_xy(first_tap['ORIG_LONGITUDE'], first_tap['ORIG_LATITUDE']),
    crs="EPSG:4326"
)

# Spatial join with your planning area polygons (gdf)
df_with_area = gpd.sjoin(
    df_points,
    gdf,
    how="left",
    predicate="within"
)

# Result has planning_area assigned to each origin point
df_with_area.head()

,CRD_NUM,BIZ_DT,BUS_SVC_NUM,DEST_LOC_ID_NUM,ENTRY_DT,ENTRY_TM,EXIT_DT,EXIT_TM,JRNY_ID_NUM,ORIG_LOC_ID_NUM,...,ORIG_LATITUDE,ORIG_LONGITUDE,ORIG_Travel_Type,next_orig_lat,next_orig_lon,next_orig_station,walk_distance,geometry,index_right,planning_area
0,100005879220,2025-02-11,NaN,22,2025-02-11,2025-02-11 11:20:34,2025-02-11,2025-02-11 11:32:25,110710909992,44,...,1.440589,103.800990,TRAIN,1.429443,103.835005,Yishun,0.0,POINT (103.80099 1.44059),11.0,WOODLANDS
1,100006223599,2025-02-11,45.0,3968,2025-02-11,2025-02-11 06:56:13,2025-02-11,2025-02-11 07:07:33,110710081977,4009,...,1.350814,103.889844,BUS,1.349708,103.873575,Serangoon NEL,195.0,POINT (103.88984 1.35081),33.0,HOUGANG
2,130013244516,2025-02-11,382.0,5960,2025-02-11,2025-02-11 10:07:29,2025-02-11,2025-02-11 10:11:38,110710535706,5953,...,1.410690,103.897592,BUS,1.408195,103.905669,One Punggol,123.0,POINT (103.89759 1.41069),18.0,PUNGGOL
3,130013244638,2025-02-11,871.0,2782,2025-02-11,2025-02-11 06:40:58,2025-02-11,2025-02-11 06:54:37,110711365561,6011,...,1.356473,103.734424,BUS,1.358612,103.751791,Bukit Gombak,248.0,POINT (103.73442 1.35647),38.0,TENGAH
4,138000455610,2025-02-11,NaN,48,2025-02-11,2025-02-11 07:19:34,2025-02-11,2025-02-11 07:28:26,110711527255,29,...,1.358612,103.751791,TRAIN,1.397535,103.747405,Yew Tee,0.0,POINT (103.75179 1.35861),2.0,BUKIT BATOK


In [74]:
df_with_area[['ORIG_LATITUDE', 'ORIG_LONGITUDE']].isna().sum()

ORIG_LATITUDE     93
ORIG_LONGITUDE    93
dtype: int64

In [75]:
df_with_area[['planning_area']].isna().sum()

planning_area    3874
dtype: int64

In [ ]:
df_with_area[df_with_area['planning_area'].isna()]
#insignificant NaN; head and tail is all from JB, ignore them for now

,CRD_NUM,BIZ_DT,BUS_SVC_NUM,DEST_LOC_ID_NUM,ENTRY_DT,ENTRY_TM,EXIT_DT,EXIT_TM,JRNY_ID_NUM,ORIG_LOC_ID_NUM,...,ORIG_LATITUDE,ORIG_LONGITUDE,ORIG_Travel_Type,next_orig_lat,next_orig_lon,next_orig_station,walk_distance,geometry,index_right,planning_area
143,148010186286,2025-02-11,950.0,3021,2025-02-11,2025-02-11 08:31:14,2025-02-11,2025-02-11 08:37:12,110709461833,6470,...,1.465427,103.768267,BUS,1.442889,103.769338,W'lands Train Checkpt,756.0,POINT (103.76827 1.46543),NaN,NaN
171,148010835293,2025-02-11,170.0,3021,2025-02-11,2025-02-11 11:14:09,2025-02-11,2025-02-11 11:17:38,110709332231,6470,...,1.465427,103.768267,BUS,1.446941,103.769253,Board W'lands CP,0.0,POINT (103.76827 1.46543),NaN,NaN
266,148012040594,2025-02-11,804.0,6553,2025-02-11,2025-02-11 05:22:09,2025-02-11,2025-02-11 05:32:59,110710600241,-99,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (NaN NaN),NaN,NaN
504,170008860626,2025-02-11,170.0,3021,2025-02-11,2025-02-11 15:10:39,2025-02-11,2025-02-11 15:17:32,110709344898,6470,...,1.465427,103.768267,BUS,1.446941,103.769253,Board W'lands CP,0.0,POINT (103.76827 1.46543),NaN,NaN
556,170008883733,2025-02-11,950.0,3021,2025-02-11,2025-02-11 16:49:26,2025-02-11,2025-02-11 16:53:44,110709540116,6470,...,1.465427,103.768267,BUS,1.446941,103.769253,Board W'lands CP,0.0,POINT (103.76827 1.46543),NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1211606,9722026580771,2025-02-11,950.0,3021,2025-02-11,2025-02-11 09:52:32,2025-02-11,2025-02-11 09:57:53,110709978845,6470,...,1.465427,103.768267,BUS,1.446941,103.769253,Board W'lands CP,0.0,POINT (103.76827 1.46543),NaN,NaN
1211695,9722026808091,2025-02-11,170.0,3021,2025-02-11,2025-02-11 07:35:40,2025-02-11,2025-02-11 07:52:11,110710156566,6470,...,1.465427,103.768267,BUS,NaN,NaN,NaN,NaN,POINT (103.76827 1.46543),NaN,NaN
1212421,9722028485397,2025-02-11,950.0,3021,2025-02-11,2025-02-11 05:32:14,2025-02-11,2025-02-11 05:43:35,110710380330,6470,...,1.465427,103.768267,BUS,1.446941,103.769253,Board W'lands CP,0.0,POINT (103.76827 1.46543),NaN,NaN
1212722,9722028953561,2025-02-11,160.0,3021,2025-02-11,2025-02-11 07:31:53,2025-02-11,2025-02-11 07:48:27,110710863413,6470,...,1.465427,103.768267,BUS,1.446941,103.769253,Board W'lands CP,0.0,POINT (103.76827 1.46543),NaN,NaN


In [ ]:
# verification
# df_with_area[df_with_area['planning_area'].isna()].tail(10)

,CRD_NUM,BIZ_DT,BUS_SVC_NUM,DEST_LOC_ID_NUM,ENTRY_DT,ENTRY_TM,EXIT_DT,EXIT_TM,JRNY_ID_NUM,ORIG_LOC_ID_NUM,...,ORIG_LATITUDE,ORIG_LONGITUDE,ORIG_Travel_Type,next_orig_lat,next_orig_lon,next_orig_station,walk_distance,geometry,index_right,planning_area
1208709,9710034511302,2025-02-11,160.0,3021,2025-02-11,2025-02-11 13:06:17,2025-02-11,2025-02-11 13:12:00,110711344045,6470,...,1.465427,103.768267,BUS,1.446941,103.769253,Board W'lands CP,0.0,POINT (103.76827 1.46543),NaN,NaN
1209549,9722016442534,2025-02-11,950.0,3021,2025-02-11,2025-02-11 13:14:40,2025-02-11,2025-02-11 13:20:30,110710469869,6470,...,1.465427,103.768267,BUS,1.446941,103.769253,Board W'lands CP,0.0,POINT (103.76827 1.46543),NaN,NaN
1209684,9722017841172,2025-02-11,170.0,3021,2025-02-11,2025-02-11 06:47:56,2025-02-11,2025-02-11 07:04:55,110709960140,6470,...,1.465427,103.768267,BUS,1.442889,103.769338,W'lands Train Checkpt,756.0,POINT (103.76827 1.46543),NaN,NaN
1209705,9722018044211,2025-02-11,170.0,3021,2025-02-11,2025-02-11 07:24:31,2025-02-11,2025-02-11 07:48:01,110710048383,6470,...,1.465427,103.768267,BUS,1.442889,103.769338,W'lands Train Checkpt,756.0,POINT (103.76827 1.46543),NaN,NaN
1210086,9722020397664,2025-02-11,160.0,3021,2025-02-11,2025-02-11 14:12:42,2025-02-11,2025-02-11 14:17:08,110709363160,6470,...,1.465427,103.768267,BUS,1.446941,103.769253,Board W'lands CP,0.0,POINT (103.76827 1.46543),NaN,NaN
1211606,9722026580771,2025-02-11,950.0,3021,2025-02-11,2025-02-11 09:52:32,2025-02-11,2025-02-11 09:57:53,110709978845,6470,...,1.465427,103.768267,BUS,1.446941,103.769253,Board W'lands CP,0.0,POINT (103.76827 1.46543),NaN,NaN
1211695,9722026808091,2025-02-11,170.0,3021,2025-02-11,2025-02-11 07:35:40,2025-02-11,2025-02-11 07:52:11,110710156566,6470,...,1.465427,103.768267,BUS,NaN,NaN,NaN,NaN,POINT (103.76827 1.46543),NaN,NaN
1212421,9722028485397,2025-02-11,950.0,3021,2025-02-11,2025-02-11 05:32:14,2025-02-11,2025-02-11 05:43:35,110710380330,6470,...,1.465427,103.768267,BUS,1.446941,103.769253,Board W'lands CP,0.0,POINT (103.76827 1.46543),NaN,NaN
1212722,9722028953561,2025-02-11,160.0,3021,2025-02-11,2025-02-11 07:31:53,2025-02-11,2025-02-11 07:48:27,110710863413,6470,...,1.465427,103.768267,BUS,1.446941,103.769253,Board W'lands CP,0.0,POINT (103.76827 1.46543),NaN,NaN
1212915,9722029326610,2025-02-11,170.0,3021,2025-02-11,2025-02-11 18:03:13,2025-02-11,2025-02-11 18:10:25,110709791554,6470,...,1.465427,103.768267,BUS,1.442889,103.769338,W'lands Train Checkpt,756.0,POINT (103.76827 1.46543),NaN,NaN


In [79]:
df_with_area[df_with_area['planning_area'].isna()].head(10)

,CRD_NUM,BIZ_DT,BUS_SVC_NUM,DEST_LOC_ID_NUM,ENTRY_DT,ENTRY_TM,EXIT_DT,EXIT_TM,JRNY_ID_NUM,ORIG_LOC_ID_NUM,...,ORIG_LATITUDE,ORIG_LONGITUDE,ORIG_Travel_Type,next_orig_lat,next_orig_lon,next_orig_station,walk_distance,geometry,index_right,planning_area
143,148010186286,2025-02-11,950.0,3021,2025-02-11,2025-02-11 08:31:14,2025-02-11,2025-02-11 08:37:12,110709461833,6470,...,1.465427,103.768267,BUS,1.442889,103.769338,W'lands Train Checkpt,756.0,POINT (103.76827 1.46543),NaN,NaN
171,148010835293,2025-02-11,170.0,3021,2025-02-11,2025-02-11 11:14:09,2025-02-11,2025-02-11 11:17:38,110709332231,6470,...,1.465427,103.768267,BUS,1.446941,103.769253,Board W'lands CP,0.0,POINT (103.76827 1.46543),NaN,NaN
266,148012040594,2025-02-11,804.0,6553,2025-02-11,2025-02-11 05:22:09,2025-02-11,2025-02-11 05:32:59,110710600241,-99,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (NaN NaN),NaN,NaN
504,170008860626,2025-02-11,170.0,3021,2025-02-11,2025-02-11 15:10:39,2025-02-11,2025-02-11 15:17:32,110709344898,6470,...,1.465427,103.768267,BUS,1.446941,103.769253,Board W'lands CP,0.0,POINT (103.76827 1.46543),NaN,NaN
556,170008883733,2025-02-11,950.0,3021,2025-02-11,2025-02-11 16:49:26,2025-02-11,2025-02-11 16:53:44,110709540116,6470,...,1.465427,103.768267,BUS,1.446941,103.769253,Board W'lands CP,0.0,POINT (103.76827 1.46543),NaN,NaN
747,170008959887,2025-02-11,160.0,3021,2025-02-11,2025-02-11 17:18:36,2025-02-11,2025-02-11 17:25:54,110709697761,6470,...,1.465427,103.768267,BUS,1.446941,103.769253,Board W'lands CP,0.0,POINT (103.76827 1.46543),NaN,NaN
930,170011329017,2025-02-11,170.0,3021,2025-02-11,2025-02-11 05:57:32,2025-02-11,2025-02-11 06:15:15,110711257859,6470,...,1.465427,103.768267,BUS,1.442889,103.769338,W'lands Train Checkpt,756.0,POINT (103.76827 1.46543),NaN,NaN
999,170011347511,2025-02-11,170.0,3021,2025-02-11,2025-02-11 07:36:58,2025-02-11,2025-02-11 07:51:38,110712678999,6470,...,1.465427,103.768267,BUS,1.442889,103.769338,W'lands Train Checkpt,756.0,POINT (103.76827 1.46543),NaN,NaN
1002,170011348488,2025-02-11,170.0,3021,2025-02-11,2025-02-11 15:15:29,2025-02-11,2025-02-11 15:20:02,110709674047,6470,...,1.465427,103.768267,BUS,1.446941,103.769253,Board W'lands CP,0.0,POINT (103.76827 1.46543),NaN,NaN
1095,170011381286,2025-02-11,170.0,3021,2025-02-11,2025-02-11 13:31:15,2025-02-11,2025-02-11 13:35:53,110711495344,6470,...,1.465427,103.768267,BUS,1.446941,103.769253,Board W'lands CP,0.0,POINT (103.76827 1.46543),NaN,NaN


In [80]:
df_with_area = df_with_area.drop(columns=['geometry', 'index_right'])

,CRD_NUM,BIZ_DT,BUS_SVC_NUM,DEST_LOC_ID_NUM,ENTRY_DT,ENTRY_TM,EXIT_DT,EXIT_TM,JRNY_ID_NUM,ORIG_LOC_ID_NUM,...,ORIG_STATION_NAME,ORIG_MRK_ID_NUM,ORIG_LATITUDE,ORIG_LONGITUDE,ORIG_Travel_Type,next_orig_lat,next_orig_lon,next_orig_station,walk_distance,planning_area
0,100005879220,2025-02-11,NaN,22,2025-02-11,2025-02-11 11:20:34,2025-02-11,2025-02-11 11:32:25,110710909992,44,...,Admiralty,44.0,1.440589,103.800990,TRAIN,1.429443,103.835005,Yishun,0.0,WOODLANDS
1,100006223599,2025-02-11,45.0,3968,2025-02-11,2025-02-11 06:56:13,2025-02-11,2025-02-11 07:07:33,110710081977,4009,...,Blk 172,4009.0,1.350814,103.889844,BUS,1.349708,103.873575,Serangoon NEL,195.0,HOUGANG
2,130013244516,2025-02-11,382.0,5960,2025-02-11,2025-02-11 10:07:29,2025-02-11,2025-02-11 10:11:38,110710535706,5953,...,Blk 322 CP,5953.0,1.410690,103.897592,BUS,1.408195,103.905669,One Punggol,123.0,PUNGGOL
3,130013244638,2025-02-11,871.0,2782,2025-02-11,2025-02-11 06:40:58,2025-02-11,2025-02-11 06:54:37,110711365561,6011,...,Tengah CC,6011.0,1.356473,103.734424,BUS,1.358612,103.751791,Bukit Gombak,248.0,TENGAH
4,138000455610,2025-02-11,NaN,48,2025-02-11,2025-02-11 07:19:34,2025-02-11,2025-02-11 07:28:26,110711527255,29,...,Bukit Gombak,29.0,1.358612,103.751791,TRAIN,1.397535,103.747405,Yew Tee,0.0,BUKIT BATOK


In [82]:
# rename
df_with_area = df_with_area.rename(columns={'planning_area': 'house_area'})

#merge to df3 on CRD_NUM
df3 = df3.merge(
    df_with_area[['CRD_NUM', 'house_area']],  # only need CRD_NUM and house_area
    on='CRD_NUM',
    how='left'  # keeps all rows in df3, adds house_area where card_num matches
)

In [83]:
df3.head(10)

,BIZ_DT,BUS_SVC_NUM,CRD_NUM,DEST_LOC_ID_NUM,ENTRY_DT,ENTRY_TM,EXIT_DT,EXIT_TM,JRNY_ID_NUM,ORIG_LOC_ID_NUM,...,ORIG_STATION_NAME,ORIG_MRK_ID_NUM,ORIG_LATITUDE,ORIG_LONGITUDE,ORIG_Travel_Type,next_orig_lat,next_orig_lon,next_orig_station,walk_distance,house_area
0,2025-02-11,NaN,100005879220,22,2025-02-11,2025-02-11 11:20:34,2025-02-11,2025-02-11 11:32:25,110710909992,44,...,Admiralty,44.0,1.440589,103.800990,TRAIN,1.429443,103.835005,Yishun,0.0,WOODLANDS
1,2025-02-11,NaN,100005879220,44,2025-02-11,2025-02-11 22:32:59,2025-02-11,2025-02-11 22:45:03,110710523649,22,...,Yishun,22.0,1.429443,103.835005,TRAIN,NaN,NaN,NaN,NaN,WOODLANDS
2,2025-02-11,45.0,100006223599,3968,2025-02-11,2025-02-11 06:56:13,2025-02-11,2025-02-11 07:07:33,110710081977,4009,...,Blk 172,4009.0,1.350814,103.889844,BUS,1.349708,103.873575,Serangoon NEL,195.0,HOUGANG
3,2025-02-11,NaN,100006223599,109,2025-02-11,2025-02-11 07:09:17,2025-02-11,2025-02-11 07:18:09,110710081977,106,...,Serangoon NEL,106.0,1.349708,103.873575,TRAIN,1.319336,103.861570,Boon Keng,0.0,HOUGANG
4,2025-02-11,NaN,100006223599,106,2025-02-11,2025-02-11 14:51:22,2025-02-11,2025-02-11 15:03:53,110710904744,109,...,Boon Keng,109.0,1.319336,103.861570,TRAIN,1.348979,103.872774,S'Goon Stn Exit B,139.0,HOUGANG
5,2025-02-11,45.0,100006223599,4008,2025-02-11,2025-02-11 15:10:02,2025-02-11,2025-02-11 15:21:55,110710226201,3967,...,S'Goon Stn Exit B,3967.0,1.348979,103.872774,BUS,NaN,NaN,NaN,NaN,HOUGANG
6,2025-02-11,382.0,130013244516,5960,2025-02-11,2025-02-11 10:07:29,2025-02-11,2025-02-11 10:11:38,110710535706,5953,...,Blk 322 CP,5953.0,1.410690,103.897592,BUS,1.408195,103.905669,One Punggol,123.0,PUNGGOL
7,2025-02-11,382.0,130013244516,5954,2025-02-11,2025-02-11 12:15:49,2025-02-11,2025-02-11 12:21:50,110709913132,5959,...,One Punggol,5959.0,1.408195,103.905669,BUS,NaN,NaN,NaN,NaN,PUNGGOL
8,2025-02-11,871.0,130013244638,2782,2025-02-11,2025-02-11 06:40:58,2025-02-11,2025-02-11 06:54:37,110711365561,6011,...,Tengah CC,6011.0,1.356473,103.734424,BUS,1.358612,103.751791,Bukit Gombak,248.0,TENGAH
9,2025-02-11,NaN,130013244638,45,2025-02-11,2025-02-11 06:58:38,2025-02-11,2025-02-11 07:20:00,110711365561,29,...,Bukit Gombak,29.0,1.358612,103.751791,TRAIN,1.436946,103.785936,Woodlands Int B5,25.0,TENGAH


In [88]:
#check if merge was done correctly: should be ok unless origin is woodlands checkpoint which will have NaN as house_area


sample_df3 = df3[['CRD_NUM', 'house_area']].sample(10, random_state=42)


check_merge = sample_df3.merge(
    df_with_area[['CRD_NUM', 'house_area']],  # original column in df_with_area
    on='CRD_NUM',
    how='left'
)


check_merge = check_merge.rename(columns={'house_area': 'original_house_area'})

print(check_merge)

         CRD_NUM   house_area_x   house_area_y
0   230003652014  MARINE PARADE  MARINE PARADE
1  9200000923619        PUNGGOL        PUNGGOL
2   188200518330    BUKIT BATOK    BUKIT BATOK
3   248200441128       CLEMENTI       CLEMENTI
4   230006221552    BUKIT TIMAH    BUKIT TIMAH
5  9200002199341  MARINE PARADE  MARINE PARADE
6   230007903776        KALLANG        KALLANG
7  9200002827320    BUKIT TIMAH    BUKIT TIMAH
8   220003185218  CHOA CHU KANG  CHOA CHU KANG
9   230000153678  CHOA CHU KANG  CHOA CHU KANG


In [89]:
# save df3 as pickle
df3.to_pickle("df3_with_house_area.pkl")